In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Keep all paths relative so this works on any machine
DATA_DIR = Path(".")
reg_path = DATA_DIR / "Registration.csv"
xlsx_path = DATA_DIR / "Course_info.xlsx"
csv_fallback = DATA_DIR / "Course_info.csv"     # <- optional CSV fallback

# Load Registration: CSV has no extra dependencies
reg_df = pd.read_csv(reg_path)

# Load Course info:
# Prefer the Excel source (as assigned), but if openpyxl isn't installed,
# fall back to CSV so the workflow still runs.
try:
    course_df = pd.read_excel(xlsx_path)
except Exception as e:
    print(f"[info] Could not read Excel ({e}). Falling back to CSV if present…")
    if csv_fallback.exists():
        course_df = pd.read_csv(csv_fallback)
    else:
        raise

print("Registration shape:", reg_df.shape)
print("Course info shape:", course_df.shape)
print("Registration columns:", reg_df.columns.tolist())
print("Course info columns:", course_df.columns.tolist())


[info] Could not read Excel ([Errno 2] No such file or directory: 'Course_info.xlsx'). Falling back to CSV if present…
Registration shape: (4900, 3)
Course info shape: (42, 3)
Registration columns: ['Student name', 'semester new', 'coursename']
Course info columns: ['Course number', 'Course Name ', 'Course Type']


In [2]:
# --- small helpers to keep the pipeline tidy ---
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Strip/normalize whitespace in column names, e.g., 'Course Name ' -> 'Course Name'."""
    out = df.copy()
    out.columns = [re.sub(r"\s+", " ", c.strip()) for c in out.columns]
    return out

def strip_str_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Trim whitespace inside string cells to avoid false mismatches during joins."""
    out = df.copy()
    for c in out.select_dtypes(include=["object"]).columns:
        out[c] = out[c].astype(str).str.strip()
    return out

def normalize_title(s: pd.Series) -> pd.Series:
    """
    Normalize course titles for robust joining:
    - uppercase
    - remove punctuation
    - collapse spaces
    """
    s2 = s.astype(str).str.upper()
    s2 = s2.str.replace(r"[^A-Z0-9 ]+", " ", regex=True)
    s2 = s2.str.replace(r"\s+", " ", regex=True).str.strip()
    return s2

# Keep a copy to measure cleaning impact
raw_reg_df = reg_df.copy()

# 1) Standardize column names & values
reg_df = clean_columns(reg_df)
reg_df = strip_str_cols(reg_df)

# 2) Remove exact duplicates to prevent double counting in analysis/joins
reg_df = reg_df.drop_duplicates().reset_index(drop=True)

# 3) Build a normalized title key for later joining with Course info
if "coursename" not in reg_df.columns:
    raise KeyError("Expected 'coursename' column in Registration.csv")
reg_df["course_title_norm"] = normalize_title(reg_df["coursename"])

# Quick exploration outputs
print("Registration (after cleaning):", reg_df.shape)
print("Duplicates removed:", len(raw_reg_df) - len(reg_df))
print("Nulls by column:\n", reg_df.isna().sum())
print("Unique students:", reg_df["Student name"].nunique())
print("Unique normalized course titles:", reg_df["course_title_norm"].nunique())

# (Optional) persist cleaned output
reg_df.to_csv("Registration_cleaned.csv", index=False)


Registration (after cleaning): (3651, 4)
Duplicates removed: 1249
Nulls by column:
 Student name         0
semester new         0
coursename           0
course_title_norm    0
dtype: int64
Unique students: 448
Unique normalized course titles: 158


In [3]:
# Preserve original for audit
raw_course_df = course_df.copy()

# Clean column names + cell strings (fixes trailing-space issues)
course_df = clean_columns(course_df)     # e.g., "Course Name " -> "Course Name"
course_df = strip_str_cols(course_df)

# Remove duplicates (being explicit, even if none)
course_df = course_df.drop_duplicates().reset_index(drop=True)

# Create the same normalized title for compatibility with Registration
if "Course Name" not in course_df.columns and "Course Name " in course_df.columns:
    course_df = course_df.rename(columns={"Course Name ": "Course Name"})
course_df["course_title_norm"] = normalize_title(course_df["Course Name"])

# Basic QA
print("Course info (after cleaning):", course_df.shape)
print("Nulls by column:\n", course_df.isna().sum())

# (Optional) persist cleaned output
course_df.to_csv("Course_info_cleaned.csv", index=False)


Course info (after cleaning): (42, 4)
Nulls by column:
 Course number        0
Course Name          0
Course Type          0
course_title_norm    0
dtype: int64


In [4]:
# Count by normalized title to collapse variants that differ only by casing/punctuation
course_counts = (
    reg_df.groupby("course_title_norm")
          .size()
          .reset_index(name="registrations")
          .sort_values("registrations", ascending=False)
)

top_count = int(course_counts["registrations"].max())
top_titles = course_counts[course_counts["registrations"] == top_count]

print("Top registrations:")
print(top_titles.to_string(index=False))


Top registrations:
    course_title_norm  registrations
COMPUT LINEAR ALGEBRA            303


In [5]:
# First try an exact inner join on the normalized title
merged_exact = pd.merge(
    reg_df,
    course_df,
    on="course_title_norm",
    how="inner",
    suffixes=("_reg", "_info"),
)
print("Exact inner-join rows:", len(merged_exact))

# --- Enhanced mitigation ---
# Some titles use abbreviations (e.g., "COMPUT LINEAR ALGEBRA" vs "COMPUTER LINEAR ALGEBRA").
# We'll use a *prefix-aware token matcher*: treat tokens as equal if identical OR share a >=5-char prefix.

STOP = {"THE", "A", "AN", "OF", "AND", "TO", "IN", "FOR", "II", "I"}

def tokens_norm(s: str):
    return [t for t in s.split() if t and t not in STOP]

def token_equiv(a: str, b: str):
    if a == b:
        return True
    MIN = 5
    return len(a) >= MIN and len(b) >= MIN and a[:MIN] == b[:MIN]

def jaccard_with_prefix(a: str, b: str) -> float:
    A, B = tokens_norm(a), tokens_norm(b)
    if not A or not B:
        return 0.0
    matched_B = set()
    matches = 0
    for i, ta in enumerate(A):
        for j, tb in enumerate(B):
            if j in matched_B:
                continue
            if token_equiv(ta, tb):
                matches += 1
                matched_B.add(j)
                break
    union = len(set(A)) + len(set(B)) - matches
    return matches / union if union else 0.0

# Find still-unmatched titles after exact join
matched_titles = set(merged_exact["course_title_norm"].unique())
all_titles = set(reg_df["course_title_norm"].unique())
unmatched = sorted(all_titles - matched_titles)

course_titles = course_df[["course_title_norm", "Course number", "Course Name"]].drop_duplicates()

# Suggest high-similarity mappings for those unmatched titles
candidates = []
for ut in unmatched:
    best_score, best_row = 0.0, None
    for _, row in course_titles.iterrows():
        score = jaccard_with_prefix(ut, row["course_title_norm"])
        if score > best_score:
            best_score, best_row = score, row
    if best_score >= 0.8 and best_row is not None:
        candidates.append({
            "course_title_norm_reg": ut,
            "course_title_norm_info": best_row["course_title_norm"],
            "Course number": best_row["Course number"],
            "Course Name": best_row["Course Name"],
            "similarity": round(best_score, 3)
        })

cand_df = pd.DataFrame(candidates).sort_values("similarity", ascending=False)
print("Suggested mappings (top 10):")
print(cand_df.head(10).to_string(index=False))

# Apply mappings: for rows that didn't match exactly, attach the suggested course number/name
if not cand_df.empty:
    reg_aug = reg_df.merge(
        cand_df[["course_title_norm_reg", "Course number", "Course Name"]],
        left_on="course_title_norm",
        right_on="course_title_norm_reg",
        how="left"
    )
    # Only add rows where exact match didn't exist but we now have a mapping
    newly_matchable = (~reg_aug["course_title_norm"].isin(matched_titles)) & reg_aug["Course number"].notna()
    add_rows = reg_aug.loc[newly_matchable,
                           ["Student name","semester new","coursename","course_title_norm","Course number","Course Name"]].copy()
    add_rows["Course Type"] = np.nan  # type unknown from mapping alone
    merged_enhanced = pd.concat([merged_exact, add_rows], ignore_index=True, sort=False)
else:
    merged_enhanced = merged_exact.copy()

print("Enhanced inner-join rows:", len(merged_enhanced))

# Save for transparency/reproducibility
merged_exact.to_csv("Registration_x_CourseInfo_merged.csv", index=False)
merged_enhanced.to_csv("Registration_x_CourseInfo_merged_enhanced_v2.csv", index=False)


Exact inner-join rows: 1791
Suggested mappings (top 10):
               course_title_norm_reg                   course_title_norm_info Course number                              Course Name  similarity
               AMERICAN HEALT POLICY                   AMERICAN HEALTH POLICY       ARTS493                   AMERICAN HEALTH POLICY         1.0
                        ART RELIGION                         ART AND RELIGION       ARTS543                         ART AND RELIGION         1.0
               AUGUSTAN CULTRL REVOL              AUGUSTAN CULTRAL REVOLUTION       ARTS561              AUGUSTAN CULTRAL REVOLUTION         1.0
   BUSINESS GERMAN MICRO PERSPECTIVE      BUSINESS GERMAN A MICRO PERSPECTIVE       ARTS494     Business German: A Micro Perspective         1.0
               CELL BIOL AND BIOCHEM                        CELL BIOL BIOCHEM       ARTS569                   CELL. BIOL. & BIOCHEM.         1.0
                 COMM THE PRESIDENCY                  COMM AND THE PRESID